# Phase P2: Batch Size Probing & Diagnostic Sweeps
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkTinkerGreen/train_with_reversibility/blob/main/notebooks/01_batch_probe.ipynb)


In [ ]:
import os, sys, json, subprocess
SMOKE = os.environ.get("REVLLM_SMOKE", "0") == "1"
IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/thinkTinkerGreen/train_with_reversibility.git"
REPO_REF = "main"

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/revllm/data', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/runs', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/revllm/probes', exist_ok=True)

    repo_dir = "/content/train_with_reversibility"
    if not os.path.exists(repo_dir):
        print(f"Cloning {REPO_URL} into {repo_dir}...")
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    else:
        print(f"Repository already present at {repo_dir}. Fetching updates...")
        subprocess.run(["git", "-C", repo_dir, "fetch", "--all"], check=False)
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{REPO_REF}"], check=False)

    os.chdir(repo_dir)
    deps = ["hatchling", "pyyaml", "tokenizers", "huggingface-hub", "datasets", "tqdm"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
else:
    curr = os.path.abspath(os.getcwd())
    repo_dir = os.path.dirname(curr) if os.path.basename(curr) == "notebooks" else curr
    os.chdir(repo_dir)

for mod in list(sys.modules.keys()):
    if mod.startswith('scripts') or mod.startswith('revllm'):
        del sys.modules[mod]

if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
src_dir = os.path.join(repo_dir, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print(f"Current working directory: {os.getcwd()}")
print(f"Execution mode: {'Smoke (CPU/CI)' if SMOKE else 'Colab GPU'}")


In [ ]:
# Run deterministic batch size probe algorithm (exponential + binary search)
from scripts.probe import probe_arch_max_batch, run_diagnostic_sweeps

archs = ["baseline", "midpoint"] if SMOKE else ["baseline", "midpoint", "midpoint_theta", "leapfrog"]
b_max_map = {}
b_safe_map = {}
all_probes = []

for arch in archs:
    b_max, b_safe, probes = probe_arch_max_batch(arch)
    b_max_map[arch] = b_max
    b_safe_map[arch] = b_safe
    all_probes.extend(probes)

b_A = b_safe_map["baseline"]
winner_candidate = "midpoint"
b_C = b_safe_map[winner_candidate]

sweeps = run_diagnostic_sweeps(winner_candidate, b_max_map["baseline"], b_max_map[winner_candidate])

probe_output = {
    "b_A": b_A,
    "b_C": b_C,
    "b_max_map": b_max_map,
    "b_safe_map": b_safe_map,
    "probe_history": all_probes,
    "diagnostic_sweeps": sweeps,
}

os.makedirs("results", exist_ok=True)
with open("results/batch_probe.json", "w") as f:
    json.dump(probe_output, f, indent=2)

if IN_COLAB and not SMOKE:
    import shutil
    shutil.copy2("results/batch_probe.json", "/content/drive/MyDrive/revllm/probes/batch_probe.json")

print(f"Batch Probe Done! Baseline B_A={b_A}, Reversible B_C={b_C}")
